In [24]:
import sys
import os
import datetime
import logging
import json

notebook_dir = os.getcwd()
working_dir = os.path.join(notebook_dir, "..")
target_dir = os.path.join(notebook_dir, "..", "external", "data_generation")
target_dir_abs = os.path.abspath(target_dir)

sys.path.append(target_dir_abs)

os.chdir(target_dir_abs)

from utils import data_generator
from utils.constituent_building import *
from utils.conjugate import *
from utils.randomize import *
from utils.vocab_sets_dynamic import *

os.chdir(notebook_dir)

In [ ]:
class AnimateSubjectPassiveGenerator(data_generator.BenchmarkGenerator):
    def __init__(self):
        super().__init__(field="syntax", 
                         linguistics="s-selection", 
                         uid="animate_subject_passive", 
                         simple_lm_method=True, 
                         one_prefix_method=True,
                         two_prefix_method=False,
                         lexically_identical=False
                         )

        #Base Nouns & Determiners
        #First I select frequent nouns, then I divide them based on animacy
        self.all_nouns = get_all_conjunctive([("category", "N"), ("frequent", "1")])
        self.animate_nouns = get_all('animate', '1', self.all_nouns)
        self.inanimate_nouns = get_all('animate', '0', self.all_nouns)
        
        #Strictly filter proper nouns for the patient so it always safely gets a physical determiner (e.g., "the", "a")
        self.common_nouns = get_all("properNoun", "0", self.all_nouns)
        
        #Collect all determiners that can be used in the experiment (e.g., "the", "a", but not "this" or "my")
        self.all_determiners = get_all("category", "(S/(S\\NP))/N")

        # First I select the transitive verbs, i.e the verbs that can appear in the passive construction. 
        # Then I filter for the ones that can take an animate subject, since that's the construction I'm targeting. 
        # Finally, I extract only the ones that have an animate subject, so I can ensure perfect agreement when I select the patient (the one which is undergoing the action) and the bad agent later on.
        transitive_verbs = get_all("category", "(S\\NP)/NP")
        en_transitive_verbs = get_all("en", "1", transitive_verbs)
        self.animate_subject_verbs = np.extract(
            ["animate=1" in x["arg_1"] for x in en_transitive_verbs], 
            en_transitive_verbs
        )

    def make_logger(self, metadata):
        log_dir = os.path.join(working_dir, "logs")
        os.makedirs(log_dir, exist_ok=True)
        safe_time = str(datetime.datetime.now()).replace(":", "-")
        log_file = f'generation-{metadata["UID"]}-{safe_time}.log'
        logging.basicConfig(filename=os.path.join(log_dir, log_file), level=logging.DEBUG)

    def sample(self):
        
        #Randomly select a verb from the list of transitive verbs with animate subjects
        V1 = choice(self.animate_subject_verbs)
        
        # I build the patient string first, so we can ensure perfect agreement between the verb and the patient.
        # The base patient is the object of the verb, so we select a noun that matches the verb's animacy and number features in the 'arg_2' field. 
        # We also restrict the selection to common nouns to ensure that we can find a determiner for it.
        base_patient = choice(get_matches_of(V1, 'arg_2', self.common_nouns))
        patient_det = choice(get_matched_by(base_patient, 'arg_1', self.all_determiners))
        
        #I ensure perfect agreement between the verb and the patient by selecting the correct auxiliary verb based on the patient's number feature.
        if base_patient['pl'] == '1':
            aux_str = choice(["are", "were", "aren't", "weren't"])
        else:
            aux_str = choice(["is", "was", "isn't", "wasn't"])

        patient_str = f"{patient_det[0]} {base_patient[0]}".strip()


        # Now we need to select the good and bad agent. We want to ensure that the bad agent is as lexically similar as possible to the good agent, while still being inanimate. 
        # To do this, we will first select a good agent that matches the verb's animacy and number features
        # then we will apply a series of filters to the inanimate nouns to find a perfect match for all features except animacy.
        while True:
            N_agent_good = choice(get_matches_of(V1, 'arg_1', self.animate_nouns))
            
            # Count how many words are in the good agent (e.g., "senator" -> 1, "French teacher" -> 2)
            good_word_count = len(N_agent_good[0].split())
            
            # To create a perfectly fair test, the "bad" word must be grammatically identical to the "good" word in every way except animacy.
            # These lines filter the inanimate nouns to ensure the bad agent has the exact same plurality, mass/count status, proper noun status, and word count as the good agent.
            matching_inanimate = get_all("pl", N_agent_good['pl'], self.inanimate_nouns)
            matching_inanimate = get_all("mass", N_agent_good['mass'], matching_inanimate)
            matching_inanimate = get_all("properNoun", N_agent_good['properNoun'], matching_inanimate)
            
            # Extract only the inanimate nouns that have the exact same number of words
            matching_inanimate = np.extract(
                [len(x[0].split()) == good_word_count for x in matching_inanimate], 
                matching_inanimate
            )
            
            # If our strict filters return at least 1 match, we break the loop and proceed
            if len(matching_inanimate) > 0:
                N_agent_bad = choice(matching_inanimate)
                break
            # otherwise, we repeat the process and select a different good agent until we find a match for the bad agent.

        # Now get the determiner based on the perfectly matched good agent
        Det = choice(get_matched_by(N_agent_good, 'arg_1', self.all_determiners))

        v_participle = V1[0]

        prefix = f"{patient_str} {aux_str} {v_participle} by {Det[0]}".replace("  ", " ").strip()

        data = {
            "sentence_good": f"{prefix} {N_agent_good[0]}.",
            "sentence_bad": f"{prefix} {N_agent_bad[0]}.",
            "one_prefix_prefix": prefix,
            "one_prefix_word_good": N_agent_good[0],
            "one_prefix_word_bad": N_agent_bad[0]
        }
        return data, data["sentence_good"]


output_dir = os.path.join(working_dir, "dataset", "blimp_animacy_new")
os.makedirs(output_dir, exist_ok=True)

generator = AnimateSubjectPassiveGenerator()
generator.generate_paradigm(number_to_generate=10000, absolute_path=os.path.join(output_dir, f"{generator.uid}.jsonl"))  

Generating data for animate_subject_passive
100 sentences generated
200 sentences generated
300 sentences generated
400 sentences generated
500 sentences generated
600 sentences generated
700 sentences generated
800 sentences generated
900 sentences generated
1000 sentences generated
1100 sentences generated
1200 sentences generated
1300 sentences generated
1400 sentences generated
1500 sentences generated
1600 sentences generated
1700 sentences generated
1800 sentences generated
1900 sentences generated
2000 sentences generated
2100 sentences generated
2200 sentences generated
2300 sentences generated
2400 sentences generated
2500 sentences generated
2600 sentences generated
2700 sentences generated
2800 sentences generated
2900 sentences generated
3000 sentences generated
3100 sentences generated
3200 sentences generated
3300 sentences generated
3400 sentences generated
3500 sentences generated
3600 sentences generated
3700 sentences generated
3800 sentences generated
3900 sentences 

In [ ]:
class VerbAgentSselectionPassiveGenerator(data_generator.BenchmarkGenerator):
    def __init__(self):
        super().__init__(field="syntax", 
                         linguistics="s-selection", 
                         uid="verb_agent_sselection_passive", 
                         simple_lm_method=True, 
                         one_prefix_method=True,
                         two_prefix_method=False,
                         lexically_identical=False)

        self.data_fields = ["clean_sentence","corrupt_sentence","clean_prefix", "corrupt_prefix"]

        #Extract the full set of frequent common nouns, which we will use to select the patient (the argument that appears after the verb in the passive construction).
        self.all_nouns = get_all_conjunctive([("category", "N"), ("frequent", "1")])
        #I filter out the proper nouns because they often don't have physical determiners (e.g., "the", "a") that we need for our construction
        self.common_nouns = get_all("properNoun", "0", self.all_nouns)
        
        #Extracts plural nouns that BLiMP tags as human/animate.
        animate_plurals = get_all_conjunctive([
            ("animate", "1"),   
            ("pl", "1"), 
            ("properNoun", "0")
        ], self.all_nouns)
        
        #Extracts plural nouns that BLiMP tags as inanimate to use as a base.
        inanimate_plurals_raw = get_all_conjunctive([
            ("animate", "0"), 
            ("pl", "1"), 
            ("properNoun", "0")
        ], self.all_nouns)
        
        #Filter the inanimate plurals to exclude any that are tagged with "institution" or "animal" features,
        # since those could potentially be interpreted as animate in certain contexts. This ensures our inanimate set is as "pure" as possible.
        inanimate_plurals = np.extract(
            [(x["institution"] != "1") and (x["animal"] != "1") for x in inanimate_plurals_raw], 
            inanimate_plurals_raw
        )
        
        #Auto-generates the animate targets by pulling the string words (x[0]), ensuring they are single words, and capping at 100.
        self.target_animate_set = list(set([x[0] for x in animate_plurals if len(x[0].split()) == 1]))[:100]

        #I completely bypassed BLiMP here and hardcoded 87 LLM-generated and verified tokens. 
        #I removed hand tools (to kill the "instrumental agency" confound) and abstract media (to ensure pure physical semantics).
        self.target_inanimate_set = [
            # Vehicles / Machines
            "planes", "trucks", "carriages", "cars", "bikes", "bicycles",
            "ships", "canoes", "carts", "skateboards", "tractors",
            "computers", "projectors", "engines", "cranes", "bulldozers",

            # Geography / Natural Forces
            "glaciers", "rivers", "lakes", "mountains", "hills", "slopes",
            "boulders", "rocks", "stones", "waves", "storms", "tides",

            # Architecture / Structural
            "doors", "floors", "gates", "stairs", "steps", "closets",
            "couches", "chairs", "tiles", "mirrors", "windows",
            "ladders", "bricks", "walls", "pillars", "beams", "panels",
            "ceilings", "columns", "bridges", "fences",

            # Dense / Heavy Physical Objects
            "boxes", "vases", "lamps", "glasses", "cups", "plates", "dishes",
            "cables", "lights", "candles", "coins", "forks", "rugs",
            "barrels", "tanks", "crates", "anchors", "chains", "weights",
            "blocks", "slabs", "pipes", "ropes",

            # Tools / Equipment
            "bolts", "wheels",

            # Natural/Organic Mass
            "logs", "branches", "roots", "debris"
        ]

        #Fetches all determiners using standard syntactic category notation.
        self.all_determiners = get_all("category", "(S/(S\\NP))/N")

        #Fetches past-participle transitive verbs from BLiMP.
        transitive_participles = get_all_conjunctive([("category", "(S\\NP)/NP"), ("en", "1")])
        
        #Lists a set of LLM-generated and verified clean verbs that strictly select for human agents.
        strict_animate_strings = [
            "congratulated", "interviewed", "hired", "fired", "scolded", #rimproverare
            "thanked", "greeted", "mocked", "praised", "bribed", #corrompere
            "nominated", "elected", "appointed", "promoted", "demoted",
            "recruited", "mentored", "coached", "tutored", "expelled",
            "investigated", "arrested", "prosecuted", "pardoned",
            "accused", "acquitted", "convicted", "sentenced",
            "interrogated", "blackmailed", "extradited", "indicted",
            "memorized", "advised", "counseled",
            "instructed", "educated", "lectured", "consulted",
            "piloted", "stolen", "purchased", "sold", "rescued", "kidnapped",
            "escorted", "vaccinated"
        ]

        #Lists a set of LLM-generated and verified the corrupt verbs. 
        #I removed geological terms for broad physical force verbs so the corrupt sentences remain on the natural language manifold.
        pure_inanimate_strings = [
            "crushed", "flooded", "buried", "shattered", "scorched",
            "demolished", "eroded", "frozen", "fractured", "dented",
            "cracked", "collapsed", "charred", "submerged", "compressed",
            "smashed", "flattened", "obliterated", "pulverized", "wrecked",
            "battered", "pounded", "blasted", "engulfed", "warped",
            "ruptured", "inundated", "splintered", "toppled"
        ]

        #We extract the clean verbs by checking if the verb's string word (x[0]) is in our strict animate list, 
        #ensuring that all our clean verbs are strictly animate-selecting.
        self.animate_agent_verbs = list(np.extract(
            [x[0] in strict_animate_strings for x in transitive_participles], 
            transitive_participles
        ))

        # We bypass BLiMP's CSV entirely and use our raw strings
        self.pure_inanimate_participles = pure_inanimate_strings

    def make_logger(self, metadata):
        log_dir = os.path.join(os.getcwd(), "logs")
        os.makedirs(log_dir, exist_ok=True)
        safe_time = str(datetime.datetime.now()).replace(":", "-").replace(" ", "_")
        log_file = f'generation-{metadata["UID"]}-{safe_time}.log'
        logging.basicConfig(filename=os.path.join(log_dir, log_file), level=logging.DEBUG)

    def sample(self):
        
        while True:
            #We start by randomly selecting a verb that strictly selects for animate agents from our curated list.
            V_clean = random.choice(self.animate_agent_verbs)
            
            try:
                #Tries to find a grammatical subject (patient) that matches the clean verb.
                valid_patients = get_matches_of(V_clean, 'arg_2', self.common_nouns)
                if valid_patients is None or len(valid_patients) == 0:
                    continue
                base_patient = random.choice(valid_patients)
            except TypeError:
                continue 
                
            break

        #Once we have a valid verb-patient pair, we select a determiner for the patient to ensure we can build a proper noun phrase (e.g., "the box", "a chair").
        patient_det = random.choice(get_matched_by(base_patient, 'arg_1', self.all_determiners))
        
        #Matches the auxiliary verb to the plural/singular subject. 
        #I strictly excluded negative auxiliaries ("aren't") to prevent double-negation confounds.
        if base_patient['pl'] == '1':
            aux_str = random.choice(["are", "were"])
        else:
            aux_str = random.choice(["is", "was"])

        patient_str = f"{patient_det[0]} {base_patient[0]}".strip()

        agent_det = random.choice(["some", "many", "several", "the", "those", "these"])

        clean_participle = V_clean[0]
        corrupt_participle = random.choice(self.pure_inanimate_participles)

        clean_prefix = f"{patient_str} {aux_str} {clean_participle} by {agent_det}".replace("  ", " ").strip()
        corrupt_prefix = f"{patient_str} {aux_str} {corrupt_participle} by {agent_det}".replace("  ", " ").strip()

        rep_animate = random.choice(self.target_animate_set)
        rep_inanimate = random.choice(self.target_inanimate_set)

        data = {
            "clean_sentence": f"{clean_prefix} {rep_animate}.",
            "corrupt_sentence": f"{corrupt_prefix} {rep_inanimate}.",
            "clean_prefix": clean_prefix,
            "corrupt_prefix": corrupt_prefix,
            "clean_verb_selectivity": "animate-selecting",
            "corrupt_verb_selectivity": "inanimate-selecting"
        }
        
        track_sentence = f"{clean_prefix} || {corrupt_prefix}"
        return data, track_sentence

output_dir = os.path.join(working_dir, "dataset", "blimp_animacy_new")
os.makedirs(output_dir, exist_ok=True)

generator = VerbAgentSselectionPassiveGenerator()

# 1. Generate the 10,000 minimal pairs
generator.generate_paradigm(
    number_to_generate=20000, 
    absolute_path=os.path.join(output_dir, f"{generator.uid}.jsonl")
)

# 2. Export the target sets to a separate, lightweight JSON file
targets_path = os.path.join(output_dir, f"{generator.uid}_targets.json")
with open(targets_path, "w") as f:
    json.dump({
        "target_animate_set": generator.target_animate_set,
        "target_inanimate_set": generator.target_inanimate_set
    }, f, indent=4)
    
print(f"Target sets successfully exported to {targets_path}")

Generating data for verb_agent_sselection_passive
100 sentences generated
200 sentences generated
300 sentences generated
400 sentences generated
500 sentences generated
600 sentences generated
700 sentences generated
800 sentences generated
900 sentences generated
1000 sentences generated
1100 sentences generated
1200 sentences generated
1300 sentences generated
1400 sentences generated
1500 sentences generated
1600 sentences generated
1700 sentences generated
1800 sentences generated
1900 sentences generated
2000 sentences generated
2100 sentences generated
2200 sentences generated
2300 sentences generated
2400 sentences generated
2500 sentences generated
2600 sentences generated
2700 sentences generated
2800 sentences generated
2900 sentences generated
3000 sentences generated
3100 sentences generated
3200 sentences generated
3300 sentences generated
3400 sentences generated
3500 sentences generated
3600 sentences generated
3700 sentences generated
3800 sentences generated
3900 sent